# Projet 11 – Étude de marché internationale pour *La Poule qui Chante*

## 🐔 Contexte
*La Poule qui Chante* est une entreprise française spécialisée dans l’élevage et la vente de poulets bio, sous le label “Poulet Agriculture Biologique”.

Actuellement active uniquement en France, elle souhaite évaluer les **opportunités d’expansion à l’international**.

## 🎯 Objectif de l’analyse
Identifier, à l’aide d’une analyse de données multi-sources :
- des **groupes de pays** aux profils similaires,
- afin de recommander les **zones géographiques prioritaires** pour l’exportation.



Ce document correspond à la **première phase** du projet :
> **Préparation et exploration des données**  




In [532]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

### 1. Importation des données



In [533]:
# Chargement des fichiers CSV
df_dispo = pd.read_csv("DisponibiliteAlimentaire_2017.csv", encoding='utf-8')
df_pop = pd.read_csv("Population_2000_2018.csv", encoding='utf-8')

# Aperçu rapide
df_dispo.head()

,Code Domaine,Domaine,Code zone,Zone,Code Élément,Élément,Code Produit,Produit,Code année,Année,Unité,Valeur,Symbole,Description du Symbole
0,FBS,Nouveaux Bilans Alimentaire,2,Afghanistan,5511,Production,2511,Blé et produits,2017,2017,Milliers de tonnes,4281.0,S,Données standardisées
1,FBS,Nouveaux Bilans Alimentaire,2,Afghanistan,5611,Importations - Quantité,2511,Blé et produits,2017,2017,Milliers de tonnes,2302.0,S,Données standardisées
2,FBS,Nouveaux Bilans Alimentaire,2,Afghanistan,5072,Variation de stock,2511,Blé et produits,2017,2017,Milliers de tonnes,-119.0,S,Données standardisées
3,FBS,Nouveaux Bilans Alimentaire,2,Afghanistan,5911,Exportations - Quantité,2511,Blé et produits,2017,2017,Milliers de tonnes,0.0,S,Données standardisées
4,FBS,Nouveaux Bilans Alimentaire,2,Afghanistan,5301,Disponibilité intérieure,2511,Blé et produits,2017,2017,Milliers de tonnes,6701.0,S,Données standardisées


### 2. Exploration de la disponibilité alimentaire


In [534]:
# Quels sont les produits disponibles ?
df_dispo['Produit'].unique()


array(['Blé et produits', 'Riz et produits', 'Orge et produits',
       'Maïs et produits', 'Seigle et produits', 'Avoine',
       'Millet et produits', 'Sorgho et produits', 'Céréales, Autres',
       'Pommes de Terre et produits', 'Ignames', 'Racines nda',
       'Sucre, canne', 'Sucre, betterave', 'Sucre Eq Brut',
       'Edulcorants Autres', 'Miel', 'Haricots', 'Pois',
       'Légumineuses Autres et produits', 'Noix et produits', 'Soja',
       'Arachides Decortiquees', 'Graines de tournesol',
       'Graines Colza/Moutarde', 'Graines de coton', 'Coco (Incl Coprah)',
       'Sésame', 'Olives', 'Plantes Oleiferes, Autre', 'Huile de Soja',
       "Huile d'Arachide", 'Huile de Tournesol',
       'Huile de Colza&Moutarde', 'Huile Graines de Coton',
       'Huile de Palmistes', 'Huile de Palme', 'Huile de Coco',
       'Huile de Sésame', "Huile d'Olive", 'Huile de Son de Riz',
       'Huile de Germe de Maïs', 'Huil Plantes Oleif Autr',
       'Tomates et produits', 'Oignons', 'Légumes, 

In [535]:
df_dispo['Élément'].unique()

array(['Production', 'Importations - Quantité', 'Variation de stock',
       'Exportations - Quantité', 'Disponibilité intérieure',
       'Aliments pour animaux', 'Semences', 'Pertes', 'Résidus',
       'Nourriture',
       'Disponibilité alimentaire en quantité (kg/personne/an)',
       'Disponibilité alimentaire (Kcal/personne/jour)',
       'Disponibilité de protéines en quantité (g/personne/jour)',
       'Disponibilité de matière grasse en quantité (g/personne/jour)',
       'Traitement', 'Autres utilisations (non alimentaire)',
       'Alimentation pour touristes'], dtype=object)

### 2.1 Création d'un DF sur la "Viande de Volailles"

Détail des analyses :

- Production totale (en tonnes)
- Importations (en tonnes)
- Exportations (en tonnes)
- Disponibilité intérieure (en tonnes)
- Consommation par habitant (kg/hab/an)



In [536]:
# Filtrer par année et produit
annee = 2017
produit = "Viande de Volailles"

# Consommation par habitant (kg/personne/an) → kg/hab
conso = df_dispo[
    (df_dispo["Année"] == annee) &
    (df_dispo["Produit"] == produit) &
    (df_dispo["Élément"] == "Disponibilité alimentaire en quantité (kg/personne/an)")
][["Zone", "Valeur"]].rename(columns={"Valeur": "conso_volaille_kg_hab"})

# Production (milliers de tonnes)
prod = df_dispo[
    (df_dispo["Année"] == annee) &
    (df_dispo["Produit"] == produit) &
    (df_dispo["Élément"] == "Production")
][["Zone", "Valeur"]].rename(columns={"Valeur": "prod_volaille_milliers_tonnes"})

# Importations (milliers de tonnes)
imp = df_dispo[
    (df_dispo["Année"] == annee) &
    (df_dispo["Produit"] == produit) &
    (df_dispo["Élément"] == "Importations - Quantité")
][["Zone", "Valeur"]].rename(columns={"Valeur": "import_volaille_milliers_tonnes"})

# Exportations (milliers de tonnes)
exp = df_dispo[
    (df_dispo["Année"] == annee) &
    (df_dispo["Produit"] == produit) &
    (df_dispo["Élément"] == "Exportations - Quantité")
][["Zone", "Valeur"]].rename(columns={"Valeur": "export_volaille_milliers_tonnes"})

# Disponibilité intérieure (milliers de tonnes)
dispo = df_dispo[
    (df_dispo["Année"] == annee) &
    (df_dispo["Produit"] == produit) &
    (df_dispo["Élément"] == "Disponibilité intérieure")
][["Zone", "Valeur"]].rename(columns={"Valeur": "dispo_volaille_milliers_tonnes"})

# Variation de stock (milliers de tonnes)
var_stock = df_dispo[
    (df_dispo["Année"] == annee) &
    (df_dispo["Produit"] == produit) &
    (df_dispo["Élément"] == "Variation de stock")
][["Zone", "Valeur"]].rename(columns={"Valeur": "variation_stock_milliers_tonnes"})

# Fusion progressive
df_volaille = conso.merge(prod, on="Zone", how="outer") \
                   .merge(imp, on="Zone", how="outer") \
                   .merge(exp, on="Zone", how="outer") \
                   .merge(dispo, on="Zone", how="outer") \
                   .merge(var_stock, on="Zone", how="outer") \
                   .rename(columns={"Zone": "Pays"})

# Aperçu
df_volaille.head()

,Pays,conso_volaille_kg_hab,prod_volaille_milliers_tonnes,import_volaille_milliers_tonnes,export_volaille_milliers_tonnes,dispo_volaille_milliers_tonnes,variation_stock_milliers_tonnes
0,Afghanistan,1.53,28.0,29.0,NaN,57.0,0.0
1,Afrique du Sud,35.69,1667.0,514.0,63.0,2118.0,-0.0
2,Albanie,16.36,13.0,38.0,0.0,47.0,4.0
3,Algérie,6.38,275.0,2.0,0.0,277.0,0.0
4,Allemagne,19.47,1514.0,842.0,646.0,1739.0,-29.0


### 3. Exploration de la population totale par pays

In [537]:
# Exploration du fichier population
df_pop.head()

,Code Domaine,Domaine,Code zone,Zone,Code Élément,Élément,Code Produit,Produit,Code année,Année,Unité,Valeur,Symbole,Description du Symbole,Note
0,OA,Séries temporelles annuelles,2,Afghanistan,511,Population totale,3010,Population-Estimations,2000,2000,1000 personnes,20779.953,X,Sources internationales sûres,NaN
1,OA,Séries temporelles annuelles,2,Afghanistan,511,Population totale,3010,Population-Estimations,2001,2001,1000 personnes,21606.988,X,Sources internationales sûres,NaN
2,OA,Séries temporelles annuelles,2,Afghanistan,511,Population totale,3010,Population-Estimations,2002,2002,1000 personnes,22600.770,X,Sources internationales sûres,NaN
3,OA,Séries temporelles annuelles,2,Afghanistan,511,Population totale,3010,Population-Estimations,2003,2003,1000 personnes,23680.871,X,Sources internationales sûres,NaN
4,OA,Séries temporelles annuelles,2,Afghanistan,511,Population totale,3010,Population-Estimations,2004,2004,1000 personnes,24726.684,X,Sources internationales sûres,NaN


### 3.1 Filtrage de la population en 2017

In [538]:
# Filtrage de la population pour l’année 2017
df_pop_2017 = df_pop[df_pop['Année'] == 2017]

# Garde seulement les colonnes utiles + renommer
df_pop_2017 = df_pop_2017[['Zone', 'Valeur']]
df_pop_2017 = df_pop_2017.rename(columns={
    'Zone': 'Pays',
    'Valeur': 'Population_milliers'
})

# Aperçu
df_pop_2017.head()

,Pays,Population_milliers
17,Afghanistan,36296.113
36,Afrique du Sud,57009.756
55,Albanie,2884.169
74,Algérie,41389.189
93,Allemagne,82658.409


### 4. Fusion des données consommation & population

In [539]:
# Jointure OUTER avec indicateur
df_check_pop = pd.merge(df_volaille, df_pop_2017, on='Pays', how='outer', indicator='source_jointure')

# Aperçu du résultat
df_check_pop['source_jointure'].value_counts()

source_jointure
both          172
right_only     64
left_only       0
Name: count, dtype: int64

In [540]:
# Afficher les pays qui n'ont pas matché
df_check_pop[df_check_pop['source_jointure'] != 'both'][['Pays', 'source_jointure']].sort_values(by='source_jointure')

,Pays,source_jointure
5,Andorre,right_only
167,Réunion,right_only
168,Sahara occidental,right_only
169,Saint-Barthélemy,right_only
171,Saint-Marin,right_only
...,...,...
142,Palaos,right_only
143,Palestine,right_only
145,Papouasie-Nouvelle-Guinée,right_only
234,Îles Vierges britanniques,right_only


### Fusion entre données FAO volaille et population

Nous réalisons ici une jointure entre les données de volaille et les données de population.

Nous utilisons `how="left"` car nous choisissons de conserver uniquement les pays pour lesquels nous avons des données sur la volaille, notre base d’analyse principale.  
Les pays présents dans le fichier population mais absents des données volaille sont considérés comme hors périmètre et ne sont donc pas intégrés à l’analyse.

In [541]:
# Jointure finale avec la population
df_final = pd.merge(df_volaille, df_pop_2017, on='Pays', how='left')

# Aperçu
df_final.head()

,Pays,conso_volaille_kg_hab,prod_volaille_milliers_tonnes,import_volaille_milliers_tonnes,export_volaille_milliers_tonnes,dispo_volaille_milliers_tonnes,variation_stock_milliers_tonnes,Population_milliers
0,Afghanistan,1.53,28.0,29.0,NaN,57.0,0.0,36296.113
1,Afrique du Sud,35.69,1667.0,514.0,63.0,2118.0,-0.0,57009.756
2,Albanie,16.36,13.0,38.0,0.0,47.0,4.0,2884.169
3,Algérie,6.38,275.0,2.0,0.0,277.0,0.0,41389.189
4,Allemagne,19.47,1514.0,842.0,646.0,1739.0,-29.0,82658.409


In [542]:
df_final.describe()


,conso_volaille_kg_hab,prod_volaille_milliers_tonnes,import_volaille_milliers_tonnes,export_volaille_milliers_tonnes,dispo_volaille_milliers_tonnes,variation_stock_milliers_tonnes,Population_milliers
count,172.000000,168.000000,170.000000,135.000000,170.000000,169.000000,1.720000e+02
mean,20.213372,725.190476,89.529412,132.185185,687.594118,13.668639,4.284175e+04
std,15.860311,2501.457125,186.669983,513.784440,2187.184747,75.364884,1.530637e+05
min,0.130000,0.000000,0.000000,0.000000,2.000000,-119.000000,5.204500e+01
25%,6.440000,13.750000,3.000000,0.000000,30.500000,0.000000,2.874480e+03
50%,18.090000,70.000000,16.000000,3.000000,100.000000,0.000000,9.757833e+03
75%,30.037500,409.750000,81.250000,32.000000,368.250000,7.000000,3.013874e+04
max,72.310000,21914.000000,1069.000000,4223.000000,18266.000000,859.000000,1.421022e+06


In [543]:
df_final.isna().sum()

Pays                                0
conso_volaille_kg_hab               0
prod_volaille_milliers_tonnes       4
import_volaille_milliers_tonnes     2
export_volaille_milliers_tonnes    37
dispo_volaille_milliers_tonnes      2
variation_stock_milliers_tonnes     3
Population_milliers                 0
dtype: int64

In [544]:
# Afficher les pays pour lesquels les exportations de volaille sont manquantes
df_final[df_final['export_volaille_milliers_tonnes'].isna()][[
    'Pays',
    'prod_volaille_milliers_tonnes',
    'import_volaille_milliers_tonnes',
    'variation_stock_milliers_tonnes',
    'dispo_volaille_milliers_tonnes'
]]

,Pays,prod_volaille_milliers_tonnes,import_volaille_milliers_tonnes,variation_stock_milliers_tonnes,dispo_volaille_milliers_tonnes
0,Afghanistan,28.0,29.0,0.0,57.0
13,Bahamas,6.0,24.0,4.0,26.0
14,Bangladesh,249.0,0.0,-0.0,250.0
23,Burkina Faso,46.0,0.0,-0.0,46.0
26,Cabo Verde,1.0,12.0,4.0,10.0
27,Cambodge,28.0,10.0,-0.0,38.0
40,Cuba,29.0,312.0,-1.0,342.0
43,Djibouti,NaN,3.0,0.0,3.0
54,Gambie,2.0,16.0,10.0,8.0
56,Grenade,1.0,7.0,-0.0,8.0


## Estimation des exportations manquantes 

Pour certains pays, les données d’exportation de viande de volaille sont manquantes dans le fichier d’origine.

Nous allons proposer une **formule comptable simplifiée** pour estimer les exportations manquantes :

\[
\Export estimée = Production + Importations + Variation de stock - Disponibilité intérieure
\]

📌 **Attention :** cette formule ne permet pas de retrouver avec précision les valeurs d’exportation réelles pour de nombreux pays. Nous l'utilisons uniquement à des fins de **remplissage ponctuel** là où la donnée officielle est absente (`NaN`).

In [545]:
# Étape 1 – Calcul de l'export estimée avec la formule simplifiée
# (on calcule pour tous les pays, pas seulement ceux avec valeur manquante, pour pouvoir vérifier ensuite)

df_final['export_calculee_simplifiee'] = (
    df_final['prod_volaille_milliers_tonnes']
    + df_final['import_volaille_milliers_tonnes']
    + df_final['variation_stock_milliers_tonnes']
    - df_final['dispo_volaille_milliers_tonnes']
)

# Étape 2 – Comparaison avec les valeurs réelles (quand elles existent)
# Cela nous permettra de voir si la formule colle bien ou non aux données FAO

df_final['ecart_export'] = df_final['export_volaille_milliers_tonnes'] - df_final['export_calculee_simplifiee']

# Étape 3 – Liste des pays où l'écart est supérieur à 100 milliers de tonnes
# (cette vérification montre que la formule est très imprécise)

grands_ecarts = df_final[df_final['ecart_export'].abs() > 100][[
    'Pays',
    'export_volaille_milliers_tonnes',
    'export_calculee_simplifiee',
    'ecart_export',
    'prod_volaille_milliers_tonnes',
    'import_volaille_milliers_tonnes',
    'variation_stock_milliers_tonnes',
    'dispo_volaille_milliers_tonnes'
]].sort_values(by='ecart_export', key=abs, ascending=False)

# Affichage des plus gros écarts entre la valeur officielle et la valeur estimée
grands_ecarts.head(150)


# Étape 4 – Utilisation de la formule uniquement pour combler les valeurs manquantes

df_final['export_volaille_milliers_tonnes'] = df_final.apply(
    lambda x: x['export_calculee_simplifiee'] if pd.isnull(x['export_volaille_milliers_tonnes']) else x['export_volaille_milliers_tonnes'],
    axis=1
)

df_final.head()

,Pays,conso_volaille_kg_hab,prod_volaille_milliers_tonnes,import_volaille_milliers_tonnes,export_volaille_milliers_tonnes,dispo_volaille_milliers_tonnes,variation_stock_milliers_tonnes,Population_milliers,export_calculee_simplifiee,ecart_export
0,Afghanistan,1.53,28.0,29.0,0.0,57.0,0.0,36296.113,0.0,NaN
1,Afrique du Sud,35.69,1667.0,514.0,63.0,2118.0,-0.0,57009.756,63.0,0.0
2,Albanie,16.36,13.0,38.0,0.0,47.0,4.0,2884.169,8.0,-8.0
3,Algérie,6.38,275.0,2.0,0.0,277.0,0.0,41389.189,0.0,0.0
4,Allemagne,19.47,1514.0,842.0,646.0,1739.0,-29.0,82658.409,588.0,58.0


In [546]:
df_final['export_volaille_milliers_tonnes'].describe()

count     168.000000
mean      107.000000
std       463.087969
min        -7.000000
25%         0.000000
50%         1.500000
75%        17.750000
max      4223.000000
Name: export_volaille_milliers_tonnes, dtype: float64

In [547]:
df_final[df_final['export_volaille_milliers_tonnes'] < 0]

,Pays,conso_volaille_kg_hab,prod_volaille_milliers_tonnes,import_volaille_milliers_tonnes,export_volaille_milliers_tonnes,dispo_volaille_milliers_tonnes,variation_stock_milliers_tonnes,Population_milliers,export_calculee_simplifiee,ecart_export
14,Bangladesh,1.50,249.0,0.0,-1.0,250.0,-0.0,159685.424,-1.0,NaN
40,Cuba,23.72,29.0,312.0,-2.0,342.0,-1.0,11339.254,-2.0,NaN
89,Madagascar,2.87,81.0,0.0,-1.0,82.0,-0.0,25570.512,-1.0,NaN
93,Mali,2.83,48.0,1.0,-7.0,52.0,-4.0,18512.430,-7.0,NaN
101,Mozambique,3.59,92.0,24.0,-1.0,116.0,-1.0,28649.018,-1.0,NaN
108,Nouvelle-Calédonie,38.71,1.0,9.0,-2.0,11.0,-1.0,277.150,-2.0,NaN
132,République-Unie de Tanzanie,1.88,105.0,2.0,-1.0,108.0,-0.0,54660.339,-1.0,NaN
165,Zimbabwe,4.68,69.0,6.0,-1.0,76.0,-0.0,14236.595,-1.0,NaN


In [548]:
# Certaines exportations négatives sont probablement des valeurs manquante
# On les remplace par 0 car il est peu probable qu'un pays exporte une quantité négative
df_final.loc[df_final['export_volaille_milliers_tonnes'] < 0, 'export_volaille_milliers_tonnes'] = 0

In [549]:
df_final['export_volaille_milliers_tonnes'].describe()

count     168.000000
mean      107.095238
std       463.065421
min         0.000000
25%         0.000000
50%         1.500000
75%        17.750000
max      4223.000000
Name: export_volaille_milliers_tonnes, dtype: float64

In [550]:
df_final.describe()

,conso_volaille_kg_hab,prod_volaille_milliers_tonnes,import_volaille_milliers_tonnes,export_volaille_milliers_tonnes,dispo_volaille_milliers_tonnes,variation_stock_milliers_tonnes,Population_milliers,export_calculee_simplifiee,ecart_export
count,172.000000,168.000000,170.000000,168.000000,170.000000,169.000000,1.720000e+02,167.000000,134.000000
mean,20.213372,725.190476,89.529412,107.095238,687.594118,13.668639,4.284175e+04,134.544910,-33.537313
std,15.860311,2501.457125,186.669983,463.065421,2187.184747,75.364884,1.530637e+05,496.492052,168.724746
min,0.130000,0.000000,0.000000,0.000000,2.000000,-119.000000,5.204500e+01,-235.000000,-1718.000000
25%,6.440000,13.750000,3.000000,0.000000,30.500000,0.000000,2.874480e+03,0.000000,-22.000000
50%,18.090000,70.000000,16.000000,1.500000,100.000000,0.000000,9.757833e+03,3.000000,-1.000000
75%,30.037500,409.750000,81.250000,17.750000,368.250000,7.000000,3.013874e+04,51.000000,1.000000
max,72.310000,21914.000000,1069.000000,4223.000000,18266.000000,859.000000,1.421022e+06,4222.000000,239.000000


In [551]:
df_final.isna().sum().sort_values(ascending=False)

ecart_export                       38
export_calculee_simplifiee          5
prod_volaille_milliers_tonnes       4
export_volaille_milliers_tonnes     4
variation_stock_milliers_tonnes     3
import_volaille_milliers_tonnes     2
dispo_volaille_milliers_tonnes      2
Pays                                0
conso_volaille_kg_hab               0
Population_milliers                 0
dtype: int64

On a toujours des pays dont les données importantes (production, importation, disponibilité) sont manquantes, on les suuprime 

In [552]:
colonnes_critiques = [
     'prod_volaille_milliers_tonnes',
    'import_volaille_milliers_tonnes',
    'dispo_volaille_milliers_tonnes'
]

df_nan_critiques = df_final[df_final[colonnes_critiques].isna().any(axis=1)]
print(f"Nombre de pays avec NaN sur les colonnes critiques : {df_nan_critiques.shape[0]}")

df_nan_critiques.head(25)

Nombre de pays avec NaN sur les colonnes critiques : 4


,Pays,conso_volaille_kg_hab,prod_volaille_milliers_tonnes,import_volaille_milliers_tonnes,export_volaille_milliers_tonnes,dispo_volaille_milliers_tonnes,variation_stock_milliers_tonnes,Population_milliers,export_calculee_simplifiee,ecart_export
43,Djibouti,2.68,NaN,3.0,NaN,3.0,0.0,944.099,NaN,NaN
92,Maldives,13.50,NaN,12.0,NaN,12.0,0.0,496.402,NaN,NaN
113,Ouzbékistan,1.96,NaN,NaN,NaN,NaN,NaN,31959.785,NaN,NaN
130,République démocratique populaire lao,10.91,NaN,NaN,NaN,NaN,NaN,6953.035,NaN,NaN


In [553]:
# Suppression des pays dont les données critiques (production, importation, disponibilité) sont manquantes
colonnes_critiques = [
    'prod_volaille_milliers_tonnes',
    'import_volaille_milliers_tonnes',
    'dispo_volaille_milliers_tonnes'
]

df_final = df_final[~df_final[colonnes_critiques].isna().any(axis=1)]

df_final.head(170)

,Pays,conso_volaille_kg_hab,prod_volaille_milliers_tonnes,import_volaille_milliers_tonnes,export_volaille_milliers_tonnes,dispo_volaille_milliers_tonnes,variation_stock_milliers_tonnes,Population_milliers,export_calculee_simplifiee,ecart_export
0,Afghanistan,1.53,28.0,29.0,0.0,57.0,0.0,36296.113,0.0,NaN
1,Afrique du Sud,35.69,1667.0,514.0,63.0,2118.0,-0.0,57009.756,63.0,0.0
2,Albanie,16.36,13.0,38.0,0.0,47.0,4.0,2884.169,8.0,-8.0
3,Algérie,6.38,275.0,2.0,0.0,277.0,0.0,41389.189,0.0,0.0
4,Allemagne,19.47,1514.0,842.0,646.0,1739.0,-29.0,82658.409,588.0,58.0
...,...,...,...,...,...,...,...,...,...,...
167,Émirats arabes unis,43.47,48.0,433.0,94.0,412.0,-26.0,9487.203,43.0,51.0
168,Équateur,19.31,340.0,0.0,0.0,341.0,-1.0,16785.361,-2.0,2.0
169,États-Unis d'Amérique,55.68,21914.0,123.0,3692.0,18266.0,80.0,325084.756,3851.0,-159.0
170,Éthiopie,0.13,14.0,1.0,1.0,14.0,0.0,106399.924,1.0,NaN


In [554]:
df_final["Population_milliers"].sum()

np.float64(7328428.482000001)

In [555]:
df_final.isna().sum().sort_values(ascending=False)

ecart_export                       34
variation_stock_milliers_tonnes     1
export_calculee_simplifiee          1
Pays                                0
conso_volaille_kg_hab               0
prod_volaille_milliers_tonnes       0
dispo_volaille_milliers_tonnes      0
export_volaille_milliers_tonnes     0
import_volaille_milliers_tonnes     0
Population_milliers                 0
dtype: int64

In [556]:
# Affichage des pays avec une valeur manquante dans "variation_stock_milliers_tonnes"
df_final[df_final["variation_stock_milliers_tonnes"].isna()]

,Pays,conso_volaille_kg_hab,prod_volaille_milliers_tonnes,import_volaille_milliers_tonnes,export_volaille_milliers_tonnes,dispo_volaille_milliers_tonnes,variation_stock_milliers_tonnes,Population_milliers,export_calculee_simplifiee,ecart_export
122,Pérou,13.47,1465.0,60.0,1.0,1523.0,NaN,31444.298,NaN,NaN


In [557]:
# Remplacement manuel de la valeur manquante pour le Pérou
df_final.loc[df_final["Pays"] == "Pérou", "variation_stock_milliers_tonnes"] = 0.0

# Recalcul de la colonne "export_calculee_simplifiee"
df_final["export_calculee_simplifiee"] = (
    df_final["prod_volaille_milliers_tonnes"]
    + df_final["import_volaille_milliers_tonnes"]
    - df_final["dispo_volaille_milliers_tonnes"]
    - df_final["variation_stock_milliers_tonnes"]
)

In [558]:
# Vérification des valeurs manquantes dans la colonne export_calculee_simplifiee
df_final[df_final["export_calculee_simplifiee"].isna()]

df_final.describe()

,conso_volaille_kg_hab,prod_volaille_milliers_tonnes,import_volaille_milliers_tonnes,export_volaille_milliers_tonnes,dispo_volaille_milliers_tonnes,variation_stock_milliers_tonnes,Population_milliers,export_calculee_simplifiee,ecart_export
count,168.000000,168.000000,168.000000,168.000000,168.000000,168.000000,1.680000e+02,168.000000,134.000000
mean,20.521726,725.190476,90.505952,107.095238,695.690476,13.750000,4.362160e+04,106.255952,-33.537313
std,15.901410,2501.457125,187.566547,463.065421,2198.968490,75.582746,1.547882e+05,463.123798,168.724746
min,0.130000,0.000000,0.000000,0.000000,2.000000,-119.000000,5.204500e+01,-1.000000,-1718.000000
25%,6.910000,13.750000,3.000000,0.000000,32.000000,0.000000,2.911678e+03,0.000000,-22.000000
50%,18.300000,70.000000,16.000000,1.500000,105.000000,0.000000,9.815582e+03,1.000000,-1.000000
75%,30.317500,409.750000,82.500000,17.750000,372.750000,7.250000,3.013874e+04,13.250000,1.000000
max,72.310000,21914.000000,1069.000000,4223.000000,18266.000000,859.000000,1.421022e+06,4222.000000,239.000000


In [559]:
# Suppression des colonnes intuiles pour alléger le dataframe.

df_final.drop(columns=[
    'export_calculee_simplifiee',
    'ecart_export'      
], inplace=True)

df_final.info()
df_final.isna().sum()

<class 'pandas.core.frame.DataFrame'>
Index: 168 entries, 0 to 171
Data columns (total 8 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Pays                             168 non-null    object 
 1   conso_volaille_kg_hab            168 non-null    float64
 2   prod_volaille_milliers_tonnes    168 non-null    float64
 3   import_volaille_milliers_tonnes  168 non-null    float64
 4   export_volaille_milliers_tonnes  168 non-null    float64
 5   dispo_volaille_milliers_tonnes   168 non-null    float64
 6   variation_stock_milliers_tonnes  168 non-null    float64
 7   Population_milliers              168 non-null    float64
dtypes: float64(7), object(1)
memory usage: 11.8+ KB


Pays                               0
conso_volaille_kg_hab              0
prod_volaille_milliers_tonnes      0
import_volaille_milliers_tonnes    0
export_volaille_milliers_tonnes    0
dispo_volaille_milliers_tonnes     0
variation_stock_milliers_tonnes    0
Population_milliers                0
dtype: int64

In [560]:
# Création d'un taux d'autosuffisance en volaille (production / consommation estimée)
df_final['autosuffisance_volaille'] = df_final['prod_volaille_milliers_tonnes'] / df_final['dispo_volaille_milliers_tonnes']

In [561]:
df_final.describe()

,conso_volaille_kg_hab,prod_volaille_milliers_tonnes,import_volaille_milliers_tonnes,export_volaille_milliers_tonnes,dispo_volaille_milliers_tonnes,variation_stock_milliers_tonnes,Population_milliers,autosuffisance_volaille
count,168.000000,168.000000,168.000000,168.000000,168.000000,168.000000,1.680000e+02,168.000000
mean,20.521726,725.190476,90.505952,107.095238,695.690476,13.750000,4.362160e+04,0.780216
std,15.901410,2501.457125,187.566547,463.065421,2198.968490,75.582746,1.547882e+05,0.490322
min,0.130000,0.000000,0.000000,0.000000,2.000000,-119.000000,5.204500e+01,0.000000
25%,6.910000,13.750000,3.000000,0.000000,32.000000,0.000000,2.911678e+03,0.398214
50%,18.300000,70.000000,16.000000,1.500000,105.000000,0.000000,9.815582e+03,0.879533
75%,30.317500,409.750000,82.500000,17.750000,372.750000,7.250000,3.013874e+04,1.000000
max,72.310000,21914.000000,1069.000000,4223.000000,18266.000000,859.000000,1.421022e+06,3.046053


### 5. Ajout de stabilité politique et PIB par habitant

In [562]:


# Lecture du fichier PIB avec saut des lignes d’en-tête
df_pib = pd.read_csv("pib_habitant.csv", skiprows=4)

# Aperçu des données
df_pib.head()

,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
0,Aruba,ABW,PIB par habitant ($ US courants),NY.GDP.PCAP.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,28440.041688,30082.158423,30645.890602,22759.807175,26749.329609,30975.998912,35718.753119,39498.594129,NaN,NaN
1,NaN,AFE,PIB par habitant ($ US courants),NY.GDP.PCAP.CD,186.089204,186.909053,197.367547,225.400079,208.962717,226.836135,...,1528.104224,1552.073722,1507.085600,1351.591669,1562.416175,1679.327622,1571.449189,1615.396356,NaN,NaN
2,Afghanistan,AFG,PIB par habitant ($ US courants),NY.GDP.PCAP.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,525.469771,491.337221,496.602504,510.787063,356.496214,357.261153,413.757895,NaN,NaN,NaN
3,NaN,AFW,PIB par habitant ($ US courants),NY.GDP.PCAP.CD,121.936832,127.451040,133.823783,139.004980,148.545883,155.561897,...,1574.230564,1720.140092,2216.385055,2030.861659,2112.794076,2138.473153,1841.855064,1411.337029,NaN,NaN
4,Angola,AGO,PIB par habitant ($ US courants),NY.GDP.PCAP.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,2790.718869,2860.093648,2493.678844,1759.356199,2303.908127,3682.113151,2916.136633,2665.874448,NaN,NaN


In [563]:
# Création d'une table de correspondance pour faire correspondre les pays

# Liste des noms de pays utilisés dans le DataFrame principal
noms_pays_df = df_final['Pays'].unique()

# Liste des noms de pays présents dans le fichier PIB
noms_pays_pib = df_pib['Country Name'].unique()

# Identifier ceux qui ne matchent pas (pays dans df_final mais pas dans df_pib)
pays_non_trouves = sorted(set(noms_pays_df) - set(noms_pays_pib))

print("Pays sans correspondance dans le CSV PIB :", pays_non_trouves)

Pays sans correspondance dans le CSV PIB : ['Bolivie (État plurinational de)', 'Chine - RAS de Hong-Kong', 'Chine - RAS de Macao', 'Chine, Taiwan Province de', 'Chine, continentale', 'Congo', "Iran (République islamique d')", 'Kirghizistan', "Royaume-Uni de Grande-Bretagne et d'Irlande du Nord", 'République de Corée', 'République de Moldova', 'République populaire démocratique de Corée', 'République-Unie de Tanzanie', 'Slovaquie', 'Tchéquie', 'Venezuela (République bolivarienne du)', 'Yémen', 'Égypte', "États-Unis d'Amérique"]


In [564]:
# Dictionnaire de correspondance entre les noms de pays dans df_final et ceux du fichier PIB
correspondance_pays_pib = {
    "Bolivie (État plurinational de)": "Bolivie",
    "Chine - RAS de Hong-Kong": "Chine, RAS de Hong Kong",
    "Chine - RAS de Macao": "Région administrative spéciale de Macao, Chine",
    "Chine, Taiwan Province de": "Taïwan",  
    "Chine, continentale": "Chine",
    "Congo": "Congo, République du",
    "Iran (République islamique d')": "Iran, République islamique d’",
    "Kirghizistan": "République kirghize",
    "Royaume-Uni de Grande-Bretagne et d'Irlande du Nord": "Royaume-Uni",
    "République de Corée": "Corée, République de",
    "République de Moldova": "Moldova",
    "République populaire démocratique de Corée": "Corée, République démocratique de",
    "République-Unie de Tanzanie": "Tanzanie",
    "Slovaquie": "République slovaque",
    "Tchéquie": "République tchèque",
    "Venezuela (République bolivarienne du)": "Venezuela",
    "Yémen": "Yémen, Rép. du",
    "Égypte": "Égypte, République arabe d’",
    "États-Unis d'Amérique": "États-Unis"
}

In [565]:
# On applique la correction des noms de pays avant de faire la jointure avec les données du PIB
df_final["Pays"] = df_final["Pays"].replace(correspondance_pays_pib)

In [566]:
# vérification des correspondances :
pays_pib = set(df_pib["Country Name"].unique())
pays_df = set(df_final["Pays"].unique())
pays_sans_correspondance = pays_df - pays_pib

print("Pays encore sans correspondance dans le fichier PIB :", pays_sans_correspondance)

Pays encore sans correspondance dans le fichier PIB : {'Taïwan'}


In [567]:
# 🔎 On vérifie si tous les pays de df_final sont bien présents dans le fichier PIB


pays_df_final = set(df_final["Pays"].unique())
pays_df_pib = set(df_pib["Country Name"].unique())

pays_non_match = sorted(pays_df_final - pays_df_pib)

print("✅ Pays encore sans correspondance dans le CSV PIB :", pays_non_match)

✅ Pays encore sans correspondance dans le CSV PIB : ['Taïwan']


In [568]:
# Ne garder que les colonnes Pays et 2017
df_pib_2017 = df_pib[['Country Name', '2017']].copy()

# Renommer les colonnes pour correspondre à df_merge
df_pib_2017.rename(columns={'Country Name': 'Pays', '2017': 'PIB_hab_USD'}, inplace=True)

# Suppression des pays avec valeurs manquantes
df_pib_2017.dropna(subset=['PIB_hab_USD'], inplace=True)

# verif
df_pib_2017.head()

,Pays,PIB_hab_USD
0,Aruba,28440.041688
1,NaN,1528.104224
2,Afghanistan,525.469771
3,NaN,1574.230564
4,Angola,2790.718869


In [569]:
# Suppression des lignes sans nom de pays
df_pib_2017 = df_pib_2017[df_pib_2017['Pays'].notna()].reset_index(drop=True)

# Vérification
df_pib_2017.head()

,Pays,PIB_hab_USD
0,Aruba,28440.041688
1,Afghanistan,525.469771
2,Angola,2790.718869
3,Albanie,5006.360130
4,Andorre,40672.971742


In [570]:
# On merge en utilisant les noms corrigés
df_final = df_final.merge(df_pib_2017, how="left", left_on="Pays", right_on="Pays")

df_final.head()

,Pays,conso_volaille_kg_hab,prod_volaille_milliers_tonnes,import_volaille_milliers_tonnes,export_volaille_milliers_tonnes,dispo_volaille_milliers_tonnes,variation_stock_milliers_tonnes,Population_milliers,autosuffisance_volaille,PIB_hab_USD
0,Afghanistan,1.53,28.0,29.0,0.0,57.0,0.0,36296.113,0.491228,525.469771
1,Afrique du Sud,35.69,1667.0,514.0,63.0,2118.0,-0.0,57009.756,0.787063,6618.335083
2,Albanie,16.36,13.0,38.0,0.0,47.0,4.0,2884.169,0.276596,5006.360130
3,Algérie,6.38,275.0,2.0,0.0,277.0,0.0,41389.189,0.992780,4554.667540
4,Allemagne,19.47,1514.0,842.0,646.0,1739.0,-29.0,82658.409,0.870615,45553.934150


In [571]:
# 1. Lecture du fichier avec skiprows pour ignorer les lignes d’en-tête inutiles
df_stab = pd.read_csv("stabilite_politique.csv", skiprows=4)

# 2. Sélection de la colonne 2017
df_stab_2017 = df_stab[["Country Name", "2017"]].copy()

# 3. Renommage des colonnes
df_stab_2017.columns = ["Pays", "Stabilite_politique"]

# 4. Suppression des lignes sans nom de pays
df_stab_2017 = df_stab_2017.dropna(subset=["Pays"])

df_stab_2017.head()

,Pays,Stabilite_politique
0,Aruba,1.313846
2,Afghanistan,-2.794976
4,Angola,-0.387895
5,Albanie,0.373771
6,Andorre,1.392890


In [572]:
# On merge en utilisant les noms corrigés
df_final = df_final.merge(df_stab_2017, how="left", left_on="Pays", right_on="Pays")

df_final.head()

,Pays,conso_volaille_kg_hab,prod_volaille_milliers_tonnes,import_volaille_milliers_tonnes,export_volaille_milliers_tonnes,dispo_volaille_milliers_tonnes,variation_stock_milliers_tonnes,Population_milliers,autosuffisance_volaille,PIB_hab_USD,Stabilite_politique
0,Afghanistan,1.53,28.0,29.0,0.0,57.0,0.0,36296.113,0.491228,525.469771,-2.794976
1,Afrique du Sud,35.69,1667.0,514.0,63.0,2118.0,-0.0,57009.756,0.787063,6618.335083,-0.284804
2,Albanie,16.36,13.0,38.0,0.0,47.0,4.0,2884.169,0.276596,5006.360130,0.373771
3,Algérie,6.38,275.0,2.0,0.0,277.0,0.0,41389.189,0.992780,4554.667540,-0.919614
4,Allemagne,19.47,1514.0,842.0,646.0,1739.0,-29.0,82658.409,0.870615,45553.934150,0.574381


## Résumé des étapes de préparation des données

- Importation et nettoyage des différentes sources de données
- Gestion des valeurs manquantes :
  - Remplissage des valeurs manquantes pour l’exportation de volaille
  - Suppression des pays avec des données manquantes (si non récupérables)
- Finalisation d’un DataFrame propre comprenant **168 pays** avec **0 valeur manquante** sur les colonnes clés4


In [573]:
df_final.describe()

,conso_volaille_kg_hab,prod_volaille_milliers_tonnes,import_volaille_milliers_tonnes,export_volaille_milliers_tonnes,dispo_volaille_milliers_tonnes,variation_stock_milliers_tonnes,Population_milliers,autosuffisance_volaille,PIB_hab_USD,Stabilite_politique
count,168.000000,168.000000,168.000000,168.000000,168.000000,168.000000,1.680000e+02,168.000000,166.000000,165.000000
mean,20.521726,725.190476,90.505952,107.095238,695.690476,13.750000,4.362160e+04,0.780216,14424.132534,-0.076671
std,15.901410,2501.457125,187.566547,463.065421,2198.968490,75.582746,1.547882e+05,0.490322,19461.439954,0.899456
min,0.130000,0.000000,0.000000,0.000000,2.000000,-119.000000,5.204500e+01,0.000000,432.324026,-2.934317
25%,6.910000,13.750000,3.000000,0.000000,32.000000,0.000000,2.911678e+03,0.398214,1976.296134,-0.629498
50%,18.300000,70.000000,16.000000,1.500000,105.000000,0.000000,9.815582e+03,0.879533,6060.687902,-0.039427
75%,30.317500,409.750000,82.500000,17.750000,372.750000,7.250000,3.013874e+04,1.000000,17776.432104,0.638511
max,72.310000,21914.000000,1069.000000,4223.000000,18266.000000,859.000000,1.421022e+06,3.046053,110193.213797,1.561946


In [574]:
df_final = df_final.dropna()

df_final.isna().sum()

Pays                               0
conso_volaille_kg_hab              0
prod_volaille_milliers_tonnes      0
import_volaille_milliers_tonnes    0
export_volaille_milliers_tonnes    0
dispo_volaille_milliers_tonnes     0
variation_stock_milliers_tonnes    0
Population_milliers                0
autosuffisance_volaille            0
PIB_hab_USD                        0
Stabilite_politique                0
dtype: int64

In [575]:
# Export du DataFrame final nettoyé pour l'utiliser dans le notebook d'exploration
df_final.to_csv("df_final.csv", index=False)